# Capítulo 1 — Teorema de Bayes como processo de aprendizado

Exemplos 1.3.4 e 1.4.1 de Phil Gregory, *Bayesian Logical Data Analysis for the Physical Sciences*.

## Objetivos

- distinguir prior, likelihood, evidência e posterior;
- comparar hipóteses discretas por meio das odds;
- visualizar o efeito da taxa-base em um teste diagnóstico;
- alterar as hipóteses do problema e interpretar o resultado.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

# Localiza a raiz do repositório mesmo se o notebook for aberto por sua pasta.
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / 'src' / 'analise_bayesiana').is_dir():
        ROOT = candidate
        break
else:
    raise FileNotFoundError('A raiz do repositório não foi encontrada.')

sys.path.insert(0, str(ROOT / 'src'))

from analise_bayesiana import (
    gaussian_pdf,
    posterior_disease_given_positive,
    posterior_discrete,
)

FIGURES_DIR = ROOT / 'figures' / 'generated'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

plt.style.use('seaborn-v0_8-whitegrid')

## Exemplo 1.3.4 — comparação entre dois modelos

Dois modelos fazem previsões pontuais para a distância de uma estrela: $d_1=100\,\mathrm{ly}$ e $d_2=200\,\mathrm{ly}$. A medida é $d=120\,\mathrm{ly}$ e o erro observacional é Gaussiano, com $\sigma=40\,\mathrm{ly}$. Inicialmente, os modelos têm probabilidades iguais.

Para $i=1,2$,

$$
p(D\mid M_i,I)=\frac{1}{\sqrt{2\pi}\sigma}
\exp\left[-\frac{(d-d_i)^2}{2\sigma^2}\right].
$$

A comparação é feita pelas odds posteriores,

$$
O_{12}=\frac{p(M_1\mid D,I)}{p(M_2\mid D,I)}
=\frac{p(M_1\mid I)}{p(M_2\mid I)}
\frac{p(D\mid M_1,I)}{p(D\mid M_2,I)}.
$$

In [ ]:
predictions = np.array([100.0, 200.0])  # ly
measurement = 120.0                     # ly
sigma = 40.0                            # ly
priors = np.array([0.5, 0.5])

likelihoods = gaussian_pdf(measurement, predictions, sigma)
posterior, evidence = posterior_discrete(priors, likelihoods)
posterior_odds = posterior[0] / posterior[1]

print(f'p(D | M1, I) = {likelihoods[0]:.8f} ly⁻¹')
print(f'p(D | M2, I) = {likelihoods[1]:.8f} ly⁻¹')
print(f'p(D | I)     = {evidence:.8f} ly⁻¹')
print(f'p(M1 | D, I) = {posterior[0]:.4%}')
print(f'p(M2 | D, I) = {posterior[1]:.4%}')
print(f'O12           = {posterior_odds:.4f}:1')

In [ ]:
distance_grid = np.linspace(0.0, 300.0, 1000)
likelihood_grid = gaussian_pdf(
    distance_grid[None, :], predictions[:, None], sigma
)
posterior_grid = priors[:, None] * likelihood_grid
posterior_grid /= posterior_grid.sum(axis=0)

colors = ('tab:blue', 'tab:orange')
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))

for i, (prediction, color) in enumerate(zip(predictions, colors)):
    axes[0].plot(
        distance_grid, likelihood_grid[i], color=color,
        label=fr'$p(d\mid M_{i + 1},I)$; $d_{i + 1}={prediction:.0f}$ ly',
    )
    axes[0].scatter(
        measurement, likelihoods[i], color=color, s=55, zorder=3
    )
axes[0].axvline(measurement, color='black', linestyle='--', label='medida: 120 ly')
axes[0].set(xlabel='Distância medida $d$ [ly]', ylabel='Densidade de probabilidade [ly$^{-1}$]')
axes[0].set_title('Likelihood sob cada modelo')
axes[0].legend(fontsize=9)

for i, color in enumerate(colors):
    axes[1].plot(
        distance_grid, posterior_grid[i], color=color,
        label=fr'$p(M_{i + 1}\mid d,I)$',
    )
axes[1].axvline(measurement, color='black', linestyle='--', label='medida: 120 ly')
axes[1].axvline(150.0, color='0.45', linestyle=':', label='fronteira para priors iguais')
axes[1].scatter(
    [measurement, measurement], posterior, color=colors, s=55, zorder=3
)
axes[1].set(xlabel='Possível distância medida $d$ [ly]', ylabel='Probabilidade posterior', ylim=(-0.02, 1.02))
axes[1].set_title('Como os dados redistribuem a crença')
axes[1].legend(fontsize=9)

fig.tight_layout()
fig.savefig(FIGURES_DIR / 'exemplo_1_3_4.png', dpi=180, bbox_inches='tight')
plt.show()

### Para interpretar e modificar

1. Por que a likelihood tem unidade de $\mathrm{ly}^{-1}$, mas a posterior dos modelos é adimensional?
2. O ponto $d=150\,\mathrm{ly}$ continua sendo a fronteira se as priors deixarem de ser iguais?
3. Aumente $\sigma$ para $80\,\mathrm{ly}$. O modelo favorecido muda? A força da evidência muda?
4. Escreva aqui, com suas palavras, a diferença entre *modelo mais provável* e *modelo verdadeiro*.

## Exemplo 1.4.1 — frequência como informação

Considere uma doença rara com prevalência $p(H\mid I)=10^{-4}$. O teste tem sensibilidade $p(+\mid H,I)=0.986$ e taxa de falso positivo $p(+\mid \bar H,I)=0.023$. Após um resultado positivo,

$$
p(H\mid +,I)=
\frac{p(H\mid I)p(+\mid H,I)}
{p(H\mid I)p(+\mid H,I)+p(\bar H\mid I)p(+\mid \bar H,I)}.
$$

A frequência observada da doença entra como dado anterior — a taxa-base — e não pode ser descartada ao interpretar o teste.

In [ ]:
prevalence = 1e-4
sensitivity = 0.986
false_positive_rate = 0.023
improved_false_positive_rate = 0.005

posterior_original = posterior_disease_given_positive(
    prevalence, sensitivity, false_positive_rate
).item()
posterior_improved = posterior_disease_given_positive(
    prevalence, sensitivity, improved_false_positive_rate
).item()

print(f'Com 2,3% de falsos positivos: p(H | +, I) = {posterior_original:.4%}')
print(f'Com 0,5% de falsos positivos: p(H | +, I) = {posterior_improved:.4%}')
print(f'Razão entre as duas posteriores: {posterior_improved / posterior_original:.2f}')

In [ ]:
prevalence_grid = np.logspace(-6, -1, 600)
false_positive_rates = np.array([0.023, 0.005, 0.001])
posterior_at_base_rate = np.array([
    posterior_disease_given_positive(prevalence, sensitivity, rate).item()
    for rate in false_positive_rates
])

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))

for rate in false_positive_rates:
    curve = posterior_disease_given_positive(prevalence_grid, sensitivity, rate)
    axes[0].plot(prevalence_grid, curve, label=f'falso positivo = {rate:.1%}')
axes[0].axvline(prevalence, color='black', linestyle='--', label='prevalência = $10^{-4}$')
axes[0].set_xscale('log')
axes[0].set_yscale('log')
axes[0].set(
    xlabel=r'Prevalência $p(H\mid I)$',
    ylabel=r'Posterior $p(H\mid +,I)$',
    title='A taxa-base controla a interpretação do teste',
)
axes[0].legend(fontsize=9)

bars = axes[1].bar(
    [f'{rate:.1%}' for rate in false_positive_rates],
    100.0 * posterior_at_base_rate,
    color=('tab:red', 'tab:orange', 'tab:green'),
)
axes[1].bar_label(bars, fmt='%.2f%%', padding=3)
axes[1].set(
    xlabel='Taxa de falso positivo',
    ylabel=r'$p(H\mid +,I)$ [%]',
    title='Resultado positivo com prevalência $10^{-4}$',
)
axes[1].set_ylim(0, 1.15 * (100.0 * posterior_at_base_rate).max())

fig.tight_layout()
fig.savefig(FIGURES_DIR / 'exemplo_1_4_1.png', dpi=180, bbox_inches='tight')
plt.show()

### Para interpretar e modificar

1. Explique por que uma sensibilidade de $98.6\%$ não implica $98.6\%$ de chance de estar doente após o teste positivo.
2. Em uma população de 100 000 pessoas, calcule separadamente o número esperado de verdadeiros e falsos positivos.
3. Qual taxa de falso positivo seria necessária para que $p(H\mid +,I)=50\%$, mantendo a prevalência e a sensibilidade fixas?
4. Refaça o gráfico para prevalências típicas de outra situação física ou observacional.

## Síntese do bloco

Para hipóteses discretas e mutuamente exclusivas,

$$p(H_i\mid D,I)=\frac{p(H_i\mid I)p(D\mid H_i,I)}
{\sum_j p(H_j\mid I)p(D\mid H_j,I)}.$$

O denominador é a **evidência** $p(D\mid I)$. Ele garante a normalização e mede quão provável era observar os dados sob todo o espaço de hipóteses considerado. A inferência é sempre condicional a esse espaço e às informações reunidas em $I$.